In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
        .appName("Platinum")

        # Delta Lake
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

        # Memory (CRITICAL for 100M+ rows)
        .config("spark.driver.memory", "8g")
        .config("spark.executor.memory", "8g")

        # CPU cores
        .config("spark.driver.cores", "4")

        # Shuffle tuning
        .config("spark.sql.shuffle.partitions", "200")

        # Adaptive query optimization
        .config("spark.sql.adaptive.enabled", "true")

        # Prevent huge driver result crashes
        .config("spark.driver.maxResultSize", "2g")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [2]:
#importing modules
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import logging

In [3]:
log_dir = "../logs"
os.makedirs(log_dir, exist_ok=True)

logging.basicConfig(
    filename=os.path.join(log_dir, "04_silver_to_platinum.log"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

In [4]:
logging.info(f"Processing Silver Layer and Gold Layer to Platinum Layer")

In [5]:
df_trip_data = spark.read\
                        .format("delta") \
                        .load(r"..\02_storage_silver\yellow_taxi")

In [6]:
from pyspark.sql.functions import expr

zone_stats = (
    df_trip_data
    .groupBy("pu_location_id")
    .agg(count("*").alias("total_trips"))
)

# compute percentile threshold
percentile_value = (
    zone_stats
    .selectExpr("percentile(total_trips, 0.6) as threshold")
    .collect()[0]["threshold"]
)

top_zone_ids = [
    r.pu_location_id
    for r in (
        zone_stats
        .filter(col("total_trips") >= percentile_value)
        .select("pu_location_id")
        .collect()
    )
]

In [7]:
# ── Hourly demand at (date, hour, zone) granularity ─
demand_series = (
    df_trip_data
    .filter(col("pu_location_id").isin(top_zone_ids))
    .withColumn("trip_date", to_date("pickup_datetime"))
    .groupBy("pu_location_id", "trip_date", "pickup_hour")
    .agg(
        count("*").alias("trip_count"),
        avg("fare_amount").alias("avg_fare"),
        avg("trip_distance").alias("avg_distance"),
        avg("passenger_count").alias("avg_passengers")
    )
    # ── calendar features ──────────────────────────
    .withColumn("day_of_week", dayofweek("trip_date"))
    .withColumn("month",       month("trip_date"))
    .withColumn("trip_year",   year("trip_date"))
    .withColumn("is_weekend",
        when(col("day_of_week").isin([1, 7]), 1).otherwise(0)
    )
    .withColumn("is_rush_hour",
        when(col("pickup_hour").between(7, 10), 1)
        .when(col("pickup_hour").between(16, 19), 1)
        .otherwise(0)
    )
    .withColumn("is_night",
        when(col("pickup_hour").between(22, 23), 1)
        .when(col("pickup_hour").between(0, 5), 1)
        .otherwise(0)
    )
    .orderBy("pu_location_id", "trip_date", "pickup_hour")
)


In [9]:
# ── Lag features (per zone, ordered by time) ───────
# These are what make the global ML model work.
# lag_1h   = same zone, 1 hour ago
# lag_24h  = same zone, same hour yesterday
# lag_168h = same zone, same hour last week
from pyspark.sql.window import Window

w = Window.partitionBy("pu_location_id").orderBy("trip_date", "pickup_hour")

demand_series = (
    demand_series
    .withColumn("lag_1h",   lag("trip_count", 1).over(w))
    .withColumn("lag_24h",  lag("trip_count", 24).over(w))
    .withColumn("lag_168h", lag("trip_count", 168).over(w))
)

In [10]:
log_dir = "../logs"
os.makedirs(log_dir, exist_ok=True)
logging.basicConfig(
    filename=os.path.join(log_dir, "forecast_demand_model_training.log"),
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)

In [11]:
def write_platinum(df, name):

    path = fr"..\04_storage_platinum\{name}"

    (
        df
        .repartition(4)
        .write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .save(path)

    )

    logging.info(f"{name} written to platinum")

write_platinum(demand_series, "demand_series")